# AN2DL 2026-27 Challenge: SPAGHETTI-1 galaxy classification
**Author:** Kasra Alizadeh

Full pipeline:
1. **EDA**: class balance, train vs test appearance.
2. **Data cleaning**
   - exact duplicates (conflicting labels removed, train/test duplicates reused at inference),
   - embedding outliers (kNN distance in a pretrained feature space): non-galaxy and heavily corrupted images,
   - alien filter: cluster gallery plus removal of every image similar to a few chosen alien examples,
   - label noise: out-of-fold predictions of a quick CV model (confident-learning style filter).
3. **Training**: stratified 5-fold CV, ImageNet-21k/22k pretrained backbones (ConvNeXt-Tiny, EfficientNetV2-S),
   dihedral + photometric augmentation, synthetic "telescope fault" augmentation, MixUp/CutMix,
   class-weighted soft cross-entropy with label smoothing, AdamW + one-cycle cosine schedule, AMP, EMA.
4. **Pseudo-labelling** of confident test images and a second training round (domain adaptation to the test set).
5. **Inference**: 8-view TTA, fold and model ensemble, stage selection and per-class bias tuning on OOF weighted F1.

Every stage writes its results to the output folder, and a time budget keeps the whole run inside one Kaggle session.

In [ ]:
import os, sys, glob, random, time, json, math, hashlib, platform, warnings
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as T

import timm
from timm.utils import ModelEmaV2

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

RUN_START = time.time()

def log(msg):
    print(f"[{(time.time() - RUN_START) / 60:6.1f} min] {msg}", flush=True)

N_STAGES = 9
def stage(n, name):
    print(f"\n{'=' * 70}\nSTAGE {n}/{N_STAGES}: {name}   (elapsed {(time.time() - RUN_START) / 60:.1f} min)\n{'=' * 70}",
          flush=True)

def bar(iterable, desc, total=None, leave=True):
    # plain-text bar: visible both in the editor and in the Kaggle log of a committed run
    return tqdm(iterable, desc=desc, total=total, leave=leave, ascii=True, ncols=100,
                mininterval=10, file=sys.stdout)

log(f"torch {torch.__version__} | timm {timm.__version__}")

## 1. Configuration
`PRESET = "quick"`: one fold, one model, about 30 min on a T4, for a first leaderboard score.
`PRESET = "full"`: the complete pipeline, several hours, best score.

In [ ]:
PRESET = "full"            # "quick" | "full" | "debug"

class CFG:
    seed = 42
    img_size = 256
    n_folds = 5
    folds_to_run = "all"
    pretrained = True                      # set False if pretrained weights are not allowed
    time_budget_h = 10.5                   # Kaggle sessions stop at 12 h

    # cleaning
    clean_duplicates = True
    use_train_duplicates_for_test = True
    embed_model = "convnext_tiny.fb_in22k_ft_in1k"
    embed_size = 224
    outlier_drop_frac = 0.01               # most isolated training images removed
    near_dup_sim = 0.98                    # near-copies above this similarity always share a CV fold
    n_clusters = 30                        # visual clusters shown in the gallery
    alien_examples = []                    # ids of clear alien / non-galaxy images, e.g. ["train_01234", "test_00042"]
    alien_sim_threshold = 0.80             # cosine similarity to an example above which an image counts as alien
    alien_max_frac = 0.25                  # safety cap on the share of training images the filter may remove
    noise_filter = True
    noise_model = dict(name="convnext_tiny.fb_in22k_ft_in1k", img=224, lr=3e-4, bs=48, epochs=12)
    noise_folds = 4
    noise_p_given_max = 0.05               # p(given label) below this ...
    noise_p_pred_min = 0.70                # ... and another class predicted above this -> dropped
    noise_max_drop_frac = 0.06

    # main models
    models = [
        dict(name="convnext_tiny.fb_in22k_ft_in1k", img=256, lr=2e-4, bs=32, epochs=40),
        dict(name="tf_efficientnetv2_s.in21k_ft_in1k", img=256, lr=3e-4, bs=32, epochs=40),
    ]
    weight_decay = 0.05
    drop_path = 0.1
    label_smoothing = 0.1
    class_weight_power = 0.25              # 0 = no class weights, 0.5 = sqrt inverse frequency
    ema_epochs = 2.0                       # EMA averaging horizon, in epochs (decay adapts to steps per epoch)
    patience = 10
    grad_clip = 1.0
    corruption_p = 0.5
    mix_prob = 0.5
    mixup_alpha = 0.2
    cutmix_alpha = 1.0

    # pseudo-labelling (check the Rules tab)
    pseudo_label = True
    pseudo_conf = 0.95
    pseudo_max_ratio = 2.0                 # pseudo images per real training image, at most
    pseudo_epochs = 20

    # inference
    tta = True
    optimize_class_bias = True

if PRESET == "quick":
    CFG.folds_to_run = [0]
    CFG.models = CFG.models[:1]
    CFG.models[0]["epochs"] = 30
    CFG.noise_filter = False
    CFG.pseudo_label = False
elif PRESET == "debug":
    CFG.n_folds, CFG.folds_to_run, CFG.noise_folds = 3, [0, 1], 2
    CFG.noise_model = dict(CFG.noise_model, epochs=1)
    CFG.models = [dict(m, epochs=2) for m in CFG.models]
    CFG.pseudo_epochs = 1
    CFG.pseudo_conf = 0.0


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
seed_everything(CFG.seed)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
N_GPU = max(1, torch.cuda.device_count()) if device.type == "cuda" else 1
USE_AMP = device.type == "cuda"
NUM_WORKERS = 0 if platform.system() == "Darwin" else min(4, os.cpu_count() or 1)

OUT_DIR = ("/kaggle/working" if os.path.isdir("/kaggle/working")
           else "../../artifacts/checkpoints/kasra" if os.path.isdir("../../artifacts")
           else "./outputs")
CKPT_DIR = f"{OUT_DIR}/ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)

def hours_left():
    return CFG.time_budget_h - (time.time() - RUN_START) / 3600

print(f"preset: {PRESET} | device: {device} x{N_GPU} | AMP: {USE_AMP} | workers: {NUM_WORKERS} | out: {OUT_DIR}")

## 2. Data loading

In [ ]:
stage(1, "data loading")
LOCAL_DATA_DIR = "/Volumes/ArmorATD/Projects/ANN & DL/dataset"

def find_data_dir():
    env = os.environ.get("AN2DL_DATA")
    if env and os.path.isfile(os.path.join(env, "train.csv")):
        return env
    patterns = ["/kaggle/input/**/train.csv", f"{glob.escape(LOCAL_DATA_DIR)}/train.csv",
                "../../artifacts/dataset/**/train.csv", "../artifacts/dataset/**/train.csv",
                "../../../dataset/train.csv"]
    for p in patterns:
        hits = [h for h in glob.glob(p, recursive=True)
                if os.path.isdir(os.path.join(os.path.dirname(h), "train"))]
        if hits:
            return os.path.dirname(hits[0])
    raise FileNotFoundError("train.csv not found: attach the competition data on Kaggle, "
                            "or set LOCAL_DATA_DIR / the AN2DL_DATA env var")

DATA_DIR = find_data_dir()
log(f"DATA_DIR: {DATA_DIR}")

train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
sub_df = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
if PRESET == "debug":
    train_df = train_df.sample(min(300, len(train_df)), random_state=CFG.seed).reset_index(drop=True)
    sub_df = sub_df.head(120)

CLASS_NAMES = ["Spaghetti (disturbed)", "Gemelli (merging)", "Gnocchi (round smooth)",
               "Ravioli (in-between smooth)", "Penne (cigar)", "Fusilli sbarra (barred spiral)",
               "Fusilli stretti (tight spiral)", "Riccioli (loose spiral)",
               "Linguine (edge-on)", "Lasagna (edge-on + bulge)"]
NUM_CLASSES = 10

def load_image(path):
    return np.array(Image.open(path).convert("RGB"))

def load_many(paths, name, threads=16):
    # parallel reads: much faster on network-mounted Kaggle inputs; filled in place to avoid a second copy
    first = load_image(paths[0])
    out = np.empty((len(paths),) + first.shape, dtype=np.uint8)
    with ThreadPoolExecutor(threads) as ex:
        for k, im in enumerate(bar(ex.map(load_image, paths), f"loading {name}", total=len(paths))):
            out[k] = im
    return out

X = load_many([f"{DATA_DIR}/train/{i}.png" for i in train_df["id"]], "train")
y = train_df["label"].values.astype(int)
XT = load_many([f"{DATA_DIR}/test/{i}.png" for i in sub_df["id"]], "test")
SOURCES = [X, XT]                      # source 0 = train, source 1 = test
log(f"train: {X.shape} | test: {XT.shape}")

def make_items(src, idx):
    idx = np.asarray(idx, dtype=np.int64)
    return np.stack([np.full(len(idx), src, dtype=np.int64), idx], axis=1)

ALL_TEST_ITEMS = make_items(1, np.arange(len(XT)))

## 3. Exploratory analysis

In [ ]:
stage(2, "exploratory analysis")
counts = np.bincount(y, minlength=NUM_CLASSES)
plt.figure(figsize=(10, 3)); plt.bar(range(NUM_CLASSES), counts)
plt.xticks(range(NUM_CLASSES)); plt.title("Training class distribution"); plt.show()
for c, n in enumerate(counts):
    print(f"{c} {CLASS_NAMES[c]:32s} {n:5d}  ({n / len(y):.1%})")

def show_grid(images, titles, n_cols=8, title=None):
    if len(images) == 0:
        return
    n_rows = math.ceil(len(images) / n_cols)
    plt.figure(figsize=(2.2 * n_cols, 2.4 * n_rows))
    for k, (im, t) in enumerate(zip(images, titles)):
        plt.subplot(n_rows, n_cols, k + 1); plt.imshow(im); plt.title(t, fontsize=7); plt.axis("off")
    if title:
        plt.suptitle(title)
    plt.tight_layout(); plt.show()

def channel_stats(arr):
    f = arr.reshape(len(arr), -1, 3).astype(np.float32) / 255.0
    return np.concatenate([f.mean(1), f.std(1)], axis=1)

st_tr = np.concatenate([channel_stats(X[i:i + 512]) for i in range(0, len(X), 512)])
st_te = np.concatenate([channel_stats(XT[i:i + 512]) for i in range(0, len(XT), 512)])
stats_table = pd.DataFrame({"train mean": st_tr.mean(0), "test mean": st_te.mean(0)},
                           index=["R mean", "G mean", "B mean", "R std", "G std", "B std"])
print("Train vs test pixel statistics (a large gap means distribution shift):")
print(stats_table.round(4))

ri = np.random.choice(len(X), 16, replace=False)
show_grid([X[i] for i in ri], [f"y={y[i]}" for i in ri], title="Random TRAIN images")
rt = np.random.choice(len(XT), min(16, len(XT)), replace=False)
show_grid([XT[i] for i in rt], [sub_df.id[i] for i in rt], title="Random TEST images")

## 4. Augmentations
Galaxies have no preferred orientation, so the full dihedral group plus arbitrary rotations is used.
`TelescopeFaults` simulates the observed corruptions: colour casts, channel swaps, sensor noise,
hot/dead pixels, stripe artifacts, foreign blobs, gamma changes and vignetting.

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

class TelescopeFaults(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        self.p = p
        self.ops = [self.color_cast, self.channel_swap, self.grayscale, self.gaussian_noise,
                    self.salt_pepper, self.stripes, self.blob, self.gamma, self.vignette]

    def forward(self, x):
        if random.random() > self.p:
            return x
        x = x.as_subclass(torch.Tensor)
        for op in random.sample(self.ops, k=random.choice([1, 1, 2])):
            x = op(x)
        return x.clamp(0, 1)

    @staticmethod
    def color_cast(x):
        gain = 1 + (torch.rand(3, 1, 1) - 0.5) * 0.6
        bias = (torch.rand(3, 1, 1) - 0.5) * 0.15
        return x * gain + bias

    @staticmethod
    def channel_swap(x):
        return x[torch.randperm(3)]

    @staticmethod
    def grayscale(x):
        g = 0.299 * x[0] + 0.587 * x[1] + 0.114 * x[2]
        return g.unsqueeze(0).repeat(3, 1, 1)

    @staticmethod
    def gaussian_noise(x):
        return x + torch.randn_like(x) * random.uniform(0.02, 0.10)

    @staticmethod
    def salt_pepper(x):
        amount = random.uniform(0.002, 0.02)
        m = torch.rand(x.shape[1:])
        x = x.clone()
        x[:, m < amount / 2] = 0.0
        x[:, m > 1 - amount / 2] = 1.0
        return x

    @staticmethod
    def stripes(x):
        x = x.clone()
        _, h, w = x.shape
        for _ in range(random.randint(1, 6)):
            width = random.randint(1, 4)
            color = torch.rand(3, 1, 1)
            alpha = random.uniform(0.4, 1.0)
            if random.random() < 0.5:
                r = random.randint(0, h - width)
                x[:, r:r + width, :] = alpha * color + (1 - alpha) * x[:, r:r + width, :]
            else:
                c = random.randint(0, w - width)
                x[:, :, c:c + width] = alpha * color + (1 - alpha) * x[:, :, c:c + width]
        return x

    @staticmethod
    def blob(x):
        _, h, w = x.shape
        ry, rx = random.uniform(0.04, 0.2) * h, random.uniform(0.04, 0.2) * w
        cy, cx = random.uniform(0, h), random.uniform(0, w)
        yy, xx = torch.meshgrid(torch.arange(h), torch.arange(w), indexing="ij")
        d = ((yy - cy) / ry) ** 2 + ((xx - cx) / rx) ** 2
        mask = torch.sigmoid((1 - d) * 6).unsqueeze(0)
        color = torch.rand(3, 1, 1)
        return x * (1 - mask) + color * mask

    @staticmethod
    def gamma(x):
        return x.clamp(1e-6, 1) ** random.uniform(0.5, 2.0)

    @staticmethod
    def vignette(x):
        _, h, w = x.shape
        yy, xx = torch.meshgrid(torch.linspace(-1, 1, h), torch.linspace(-1, 1, w), indexing="ij")
        gx, gy = random.uniform(-0.6, 0.6), random.uniform(-0.6, 0.6)
        grad = 1 + gx * xx + gy * yy - random.uniform(0, 0.4) * (xx ** 2 + yy ** 2)
        return x * grad.clamp(0.2, 1.6).unsqueeze(0)

def make_train_tf(size):
    ops = [
        T.ToImage(),
        T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
        T.RandomApply([T.RandomRotation(180, interpolation=T.InterpolationMode.BILINEAR)], p=0.75),
        T.RandomResizedCrop(size, scale=(0.7, 1.0), ratio=(0.85, 1.15), antialias=True),
        T.RandomApply([T.ColorJitter(0.3, 0.3, 0.3, 0.05)], p=0.5),
        T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 2.0))], p=0.15),
    ]
    if hasattr(T, "JPEG"):
        ops.append(T.RandomApply([T.JPEG((30, 95))], p=0.15))
    ops += [
        T.ToDtype(torch.float32, scale=True),
        TelescopeFaults(p=CFG.corruption_p),
        T.Normalize(MEAN, STD),
        T.RandomErasing(p=0.25, scale=(0.02, 0.12), value=0),
    ]
    return T.Compose(ops)

def make_eval_tf(size):
    return T.Compose([
        T.ToImage(),
        T.Resize((size, size), antialias=True),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(MEAN, STD),
    ])

class ArrayDataset(Dataset):
    def __init__(self, items, labels=None, transform=None):
        self.items, self.labels, self.transform = items, labels, transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, k):
        src, i = self.items[k]
        x = self.transform(SOURCES[src][i])
        return x, int(self.labels[k]) if self.labels is not None else -1

def eval_loader(items, size, bs=64):
    return DataLoader(ArrayDataset(items, transform=make_eval_tf(size)), batch_size=bs, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=device.type == "cuda")

def denorm(t):
    return (t.permute(1, 2, 0).numpy() * np.array(STD) + np.array(MEAN)).clip(0, 1)

ds_vis = ArrayDataset(make_items(0, np.arange(8)), y[:8], make_train_tf(CFG.img_size))
show_grid([denorm(ds_vis[k % 8][0]) for k in range(24)], [f"y={y[k % 8]}" for k in range(24)],
          title="Augmented training samples")

## 5. Training utilities

In [ ]:
def weighted_f1(y_true, probs):
    return f1_score(y_true, probs.argmax(1), average="weighted")

def class_weights(labels):
    cnt = np.bincount(labels, minlength=NUM_CLASSES).astype(np.float64)
    w = (cnt.max() / np.maximum(cnt, 1)) ** CFG.class_weight_power
    return torch.tensor(w / w.mean(), dtype=torch.float32, device=device)

def soft_targets(yb):
    t = F.one_hot(yb, NUM_CLASSES).float()
    return t * (1 - CFG.label_smoothing) + CFG.label_smoothing / NUM_CLASSES

def mix_batch(x, t):
    if random.random() > CFG.mix_prob or x.size(0) < 2:
        return x, t
    perm = torch.randperm(x.size(0), device=x.device)
    if random.random() < 0.5:
        lam = float(np.random.beta(CFG.mixup_alpha, CFG.mixup_alpha))
        x = lam * x + (1 - lam) * x[perm]
    else:
        lam = float(np.random.beta(CFG.cutmix_alpha, CFG.cutmix_alpha))
        h, w = x.shape[2:]
        ch, cw = int(h * math.sqrt(1 - lam)), int(w * math.sqrt(1 - lam))
        cy, cx = random.randint(0, h - 1), random.randint(0, w - 1)
        y1, y2 = max(cy - ch // 2, 0), min(cy + ch // 2, h)
        x1, x2 = max(cx - cw // 2, 0), min(cx + cw // 2, w)
        x = x.clone()
        x[:, :, y1:y2, x1:x2] = x[perm][:, :, y1:y2, x1:x2]
        lam = 1 - (y2 - y1) * (x2 - x1) / (h * w)
    return x, lam * t + (1 - lam) * t[perm]

def soft_ce(logits, target, weight):
    logp = F.log_softmax(logits.float(), dim=1)
    return -(target * logp * weight).sum(1).mean()

def tta_views(x):
    views = []
    for base in (x, x.flip(3)):
        for k in range(4):
            views.append(torch.rot90(base, k, (2, 3)))
    return views

@torch.no_grad()
def predict(model, loader, tta=False, desc=None):
    model.eval()
    outs = []
    it = bar(loader, desc, total=len(loader), leave=False) if desc else loader
    for xb, _ in it:
        xb = xb.to(device, non_blocking=True)
        views = tta_views(xb) if tta else [xb]
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            p = sum(F.softmax(model(v).float(), dim=1) for v in views) / len(views)
        outs.append(p.cpu())
    return torch.cat(outs).numpy()

def short_name(name):
    return name.split(".")[0]

def fit(mcfg, tr_items, tr_labels, va_items, va_labels, tag, epochs, cw):
    size = mcfg.get("img", CFG.img_size)
    bs = mcfg["bs"] * N_GPU
    lr = mcfg["lr"] * math.sqrt(N_GPU)
    train_loader = DataLoader(ArrayDataset(tr_items, tr_labels, make_train_tf(size)), batch_size=bs,
                              shuffle=True, drop_last=True, num_workers=NUM_WORKERS,
                              pin_memory=device.type == "cuda", persistent_workers=NUM_WORKERS > 0)
    val_loader = eval_loader(va_items, size)

    model = timm.create_model(mcfg["name"], pretrained=CFG.pretrained, num_classes=NUM_CLASSES,
                              drop_path_rate=CFG.drop_path).to(device)
    ema_decay = 1 - 1 / max(10.0, CFG.ema_epochs * len(train_loader))
    ema = ModelEmaV2(model, decay=ema_decay)
    net = nn.DataParallel(model) if N_GPU > 1 else model
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=CFG.weight_decay)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, total_steps=max(1, epochs * len(train_loader)),
        pct_start=0.05, anneal_strategy="cos", div_factor=20, final_div_factor=100)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    ckpt = f"{CKPT_DIR}/{tag}.pth"
    best, bad, history = -1.0, 0, []
    for epoch in range(epochs):
        net.train()
        t0, run_loss = time.time(), 0.0
        for xb, yb in bar(train_loader, f"{tag} ep {epoch:02d}/{epochs}", total=len(train_loader), leave=False):
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            xb, target = mix_batch(xb, soft_targets(yb))
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
                logits = net(xb)
            loss = soft_ce(logits, target, cw)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
            scaler.step(opt); scaler.update(); sched.step()
            ema.update(model)
            run_loss += loss.item()

        val_probs = predict(ema.module, val_loader)
        f1 = weighted_f1(va_labels, val_probs)
        history.append(f1)
        mark = ""
        if f1 > best:
            best, bad, mark = f1, 0, " *"
            torch.save(ema.module.state_dict(), ckpt)
        else:
            bad += 1
        log(f"  [{tag}] ep {epoch:02d}/{epochs} loss {run_loss / max(1, len(train_loader)):.4f} "
            f"val wF1 {f1:.4f} ({time.time() - t0:.0f}s){mark}")
        if bad >= CFG.patience:
            print("  early stopping")
            break

    model.load_state_dict(torch.load(ckpt, map_location=device))
    oof = predict(model, val_loader, tta=CFG.tta)
    print(f"  [{tag}] best wF1 {best:.4f} | with TTA {weighted_f1(va_labels, oof):.4f}")
    del ema, net, opt
    return model, oof, history

def free_memory():
    if device.type == "cuda":
        torch.cuda.empty_cache()

def run_cv(mcfg, stage, pool, epochs, folds, pseudo_items=None, pseudo_labels=None, predict_test=True):
    """K-fold training on the `pool` indices of the training set.
    Pseudo-labelled test images, if given, are added to every training fold (never to validation)."""
    size = mcfg.get("img", CFG.img_size)
    oof = np.full((len(X), NUM_CLASSES), np.nan, dtype=np.float32)
    test_pred = np.zeros((len(XT), NUM_CLASSES), dtype=np.float32)
    n_done, last_fold_h, histories = 0, 0.0, {}
    skf = StratifiedGroupKFold(n_splits=CFG.n_folds if stage != "noise" else CFG.noise_folds,
                               shuffle=True, random_state=CFG.seed)
    for fold, (a, b) in enumerate(skf.split(pool, y[pool], groups=GROUPS[pool])):
        if folds != "all" and fold not in folds:
            continue
        if n_done > 0 and hours_left() < 1.3 * last_fold_h + 0.25:
            print(f"time budget reached: skipping remaining folds of {stage}/{short_name(mcfg['name'])}")
            break
        t0 = time.time()
        tr_idx, va_idx = pool[a], pool[b]
        tr_items, tr_labels = make_items(0, tr_idx), y[tr_idx]
        if pseudo_items is not None and len(pseudo_items):
            tr_items = np.concatenate([tr_items, pseudo_items])
            tr_labels = np.concatenate([tr_labels, pseudo_labels])
        tag = f"{stage}_{short_name(mcfg['name'])}_f{fold}"
        log(f"== {tag} | train {len(tr_items)} | val {len(va_idx)} | {hours_left():.2f} h left ==")
        model, fold_oof, hist = fit(mcfg, tr_items, tr_labels, make_items(0, va_idx), y[va_idx],
                                    tag, epochs, class_weights(y[tr_idx]))
        oof[va_idx] = fold_oof
        histories[fold] = hist
        if predict_test:
            log(f"  [{tag}] predicting test set")
            test_pred += predict(model, eval_loader(ALL_TEST_ITEMS, size), tta=CFG.tta, desc=f"{tag} test")
        n_done += 1
        del model
        free_memory()
        last_fold_h = (time.time() - t0) / 3600
    if n_done:
        test_pred /= n_done
    return oof, test_pred, n_done, histories

def oof_score(oof, idx=None):
    m = ~np.isnan(oof[:, 0])
    if idx is not None:
        keep = np.zeros(len(m), bool); keep[idx] = True; m &= keep
    return (weighted_f1(y[m], oof[m]) if m.any() else float("nan")), m

## 6. Data cleaning
### 6.1 Exact duplicates

In [ ]:
stage(3, "cleaning: exact duplicates")
report = {"preset": PRESET, "n_train_raw": int(len(X)), "n_test": int(len(XT))}
keep = np.ones(len(X), dtype=bool)

h_tr = np.array([hashlib.md5(im.tobytes()).hexdigest() for im in bar(X, "hashing train")])
h_te = np.array([hashlib.md5(im.tobytes()).hexdigest() for im in bar(XT, "hashing test")])
dup_df = pd.DataFrame({"hash": h_tr, "label": y, "pos": np.arange(len(X))})
n_conflict, n_redundant = 0, 0
if CFG.clean_duplicates:
    for _, g in dup_df.groupby("hash"):
        if len(g) < 2:
            continue
        if g.label.nunique() > 1:
            keep[g.pos.values] = False; n_conflict += len(g)
        else:
            keep[g.pos.values[1:]] = False; n_redundant += len(g) - 1

train_hash_label = {h: l for h, l, k in zip(h_tr, y, keep) if k}
test_dup_label = np.array([train_hash_label.get(h, -1) for h in h_te])
report.update(dup_conflicting_removed=n_conflict, dup_redundant_removed=n_redundant,
              test_exact_duplicates_of_train=int((test_dup_label >= 0).sum()))
print(f"duplicates: {n_conflict} conflicting removed, {n_redundant} redundant removed, "
      f"{(test_dup_label >= 0).sum()} test images identical to a training image")

### 6.2 Embedding outliers
Images far from all their neighbours in a pretrained feature space (train and test together) are
non-galaxy images or heavily corrupted observations.

In [ ]:
@torch.no_grad()
def embed(items, model, size, desc="embedding"):
    # full precision so that the embeddings, clusters and alien matches are reproducible between runs
    feats = []
    loader = eval_loader(items, size)
    for xb, _ in bar(loader, desc, total=len(loader)):
        feats.append(F.normalize(model(xb.to(device)).float(), dim=1).cpu())
    return torch.cat(feats)

stage(4, "cleaning: embeddings and outliers")
emb_model = timm.create_model(CFG.embed_model, pretrained=CFG.pretrained, num_classes=0).to(device).eval()
E_tr = embed(make_items(0, np.arange(len(X))), emb_model, CFG.embed_size, "embedding train")
E_te = embed(ALL_TEST_ITEMS, emb_model, CFG.embed_size, "embedding test")
del emb_model; free_memory()

E_all = torch.cat([E_tr, E_te]).to(device)
k_nn = min(10, len(E_all) - 1)
knn_sim = []
for i in range(0, len(E_all), 2048):
    s = E_all[i:i + 2048] @ E_all.T
    knn_sim.append(s.topk(k_nn + 1, dim=1).values[:, 1:].float().cpu())
knn_sim = torch.cat(knn_sim)
outlier_score = (1 - knn_sim.mean(1)).numpy()
near_dup_sim = knn_sim[:, 0].numpy()
del E_all; free_memory()

# near-duplicate groups inside the training set (connected components of very similar pairs)
E_tr_dev = E_tr.to(device)
nn_vals, nn_idx = [], []
for i in range(0, len(X), 2048):
    v, j = (E_tr_dev[i:i + 2048] @ E_tr_dev.T).topk(min(6, len(X)), dim=1)
    nn_vals.append(v[:, 1:].cpu().numpy()); nn_idx.append(j[:, 1:].cpu().numpy())
nn_vals, nn_idx = np.concatenate(nn_vals), np.concatenate(nn_idx)
del E_tr_dev; free_memory()

def near_dup_groups(thr):
    parent = np.arange(len(X))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a
    for r, c in zip(*np.where(nn_vals >= thr)):
        ra, rb = find(r), find(int(nn_idx[r, c]))
        if ra != rb:
            parent[ra] = rb
    return np.array([find(a) for a in range(len(X))])

# chained similarities must not merge large parts of a class into one group
max_group = max(10, int(0.01 * len(X)))
for thr_g in [CFG.near_dup_sim, 0.985, 0.99, 0.995, 1.01]:
    GROUPS = near_dup_groups(thr_g)
    if np.bincount(np.unique(GROUPS, return_inverse=True)[1]).max() <= max_group:
        break
report["near_dup_group_threshold"] = float(thr_g)
n_grouped = int((pd.Series(GROUPS).map(pd.Series(GROUPS).value_counts()) > 1).sum())
log(f"embeddings and outlier scores done | {n_grouped} training images in near-duplicate groups "
    f"({len(np.unique(GROUPS))} groups), kept together in CV folds")

score_tr, score_te = outlier_score[:len(X)], outlier_score[len(X):]
train_df["outlier_score"] = score_tr
n_out = int(round(CFG.outlier_drop_frac * len(X)))
out_idx = np.argsort(-score_tr)[:n_out]
keep[out_idx] = False
report.update(outliers_removed=n_out, near_duplicate_pairs_train=int((near_dup_sim[:len(X)] > CFG.near_dup_sim).sum()),
              train_images_in_near_dup_groups=n_grouped)

top_tr = np.argsort(-score_tr)[:24]
show_grid([X[i] for i in top_tr], [f"{train_df.id[i]} y={y[i]}\n{score_tr[i]:.3f}" for i in top_tr],
          title="Most isolated TRAIN images (top ones are removed)")
top_te = np.argsort(-score_te)[:24]
show_grid([XT[i] for i in top_te], [f"{sub_df.id[i]}\n{score_te[i]:.3f}" for i in top_te],
          title="Most isolated TEST images")
plt.figure(figsize=(8, 3))
plt.hist(score_tr, 60, alpha=.6, density=True, label="train")
plt.hist(score_te, 60, alpha=.6, density=True, label="test")
plt.legend(); plt.title("kNN outlier score"); plt.show()
print(f"removed {n_out} training outliers")

### 6.3 Cluster gallery and alien filter
Train and test embeddings are grouped with k-means. Each cluster is shown with its train/test counts and
mean outlier score, so groups of aliens, artifacts or test-only appearance stand out.

**How to use it:** copy the ids of a few clear alien images from the gallery into `CFG.alien_examples`
and run again. Every training image whose embedding is close to one of the examples is removed; matching
test images are excluded from pseudo-labelling. The "borderline" grid helps to tune `alien_sim_threshold`.

In [ ]:
stage(5, "cleaning: cluster gallery and alien filter")
from sklearn.cluster import KMeans

E_np = torch.cat([E_tr, E_te]).numpy()
k_cl = min(CFG.n_clusters, len(E_np))
cl = KMeans(n_clusters=k_cl, n_init=4, random_state=CFG.seed).fit_predict(E_np)
log("clustering done")
cl_tr, cl_te = cl[:len(X)], cl[len(X):]
train_df["cluster"] = cl_tr
pd.DataFrame({"id": sub_df["id"], "cluster": cl_te, "outlier_score": score_te}).to_csv(
    f"{OUT_DIR}/test_clusters.csv", index=False)

cl_summary = pd.DataFrame({
    "train": np.bincount(cl_tr, minlength=k_cl),
    "test": np.bincount(cl_te, minlength=k_cl),
    "mean_outlier": [outlier_score[cl == c].mean() for c in range(k_cl)],
})
cl_summary["test_share"] = cl_summary.test / (cl_summary.train + cl_summary.test)
cl_order = cl_summary.sort_values("mean_outlier", ascending=False).index
print("Clusters, most unusual first (test_share far from the global value = train/test shift):")
print(f"global test share: {len(XT) / (len(X) + len(XT)):.3f}")
print(cl_summary.loc[cl_order].round(3).to_string())

rng_g = np.random.RandomState(CFG.seed)
for c in cl_order:
    tr_c, te_c = np.where(cl_tr == c)[0], np.where(cl_te == c)[0]
    p_tr = rng_g.choice(tr_c, min(5, len(tr_c)), replace=False) if len(tr_c) else []
    p_te = rng_g.choice(te_c, min(8 - len(p_tr), len(te_c)), replace=False) if len(te_c) else []
    show_grid([X[i] for i in p_tr] + [XT[i] for i in p_te],
              [f"{train_df.id[i]} y={y[i]}" for i in p_tr] + [f"{sub_df.id[i]}" for i in p_te],
              title=f"cluster {c} | train {len(tr_c)} | test {len(te_c)} | "
                    f"outlier {cl_summary.mean_outlier[c]:.3f}")

alien_test = np.zeros(len(XT), dtype=bool)
if CFG.alien_examples:
    pos_tr = {v: i for i, v in enumerate(train_df["id"])}
    pos_te = {v: i for i, v in enumerate(sub_df["id"])}
    protos = [E_tr[pos_tr[i]] for i in CFG.alien_examples if i in pos_tr] + \
             [E_te[pos_te[i]] for i in CFG.alien_examples if i in pos_te]
    unknown = [i for i in CFG.alien_examples if i not in pos_tr and i not in pos_te]
    if unknown:
        print("ids not found (ignored):", unknown)
    if protos:
        P = torch.stack(protos)
        sim_tr = (E_tr @ P.T).max(1).values.numpy()
        sim_te = (E_te @ P.T).max(1).values.numpy()
        thr = CFG.alien_sim_threshold
        cap = int(CFG.alien_max_frac * len(X))
        if (sim_tr >= thr).sum() > cap:
            thr = float(np.sort(sim_tr)[::-1][cap])
            thr = np.nextafter(np.float32(thr), np.float32(2))
            print(f"WARNING: threshold {CFG.alien_sim_threshold} would remove more than "
                  f"{CFG.alien_max_frac:.0%} of the training set; raised to {thr:.3f}. "
                  f"Check the examples and the grids below.")
        alien_tr = sim_tr >= thr
        alien_test = sim_te >= thr
        keep[alien_tr] = False
        train_df["alien_sim"] = sim_tr
        report.update(aliens_removed_train=int(alien_tr.sum()), aliens_flagged_test=int(alien_test.sum()))
        print(f"alien filter: {alien_tr.sum()} training images removed, "
              f"{alien_test.sum()} test images excluded from pseudo-labelling")

        rem = np.where(alien_tr)[0]
        rem = rem[np.argsort(sim_tr[rem])][:24]
        show_grid([X[i] for i in rem], [f"{train_df.id[i]} y={y[i]}\nsim {sim_tr[i]:.2f}" for i in rem],
                  title="Removed as alien (least similar first: check these are really aliens)")
        border = np.where(~alien_tr)[0]
        border = border[np.argsort(-sim_tr[border])][:16]
        show_grid([X[i] for i in border], [f"{train_df.id[i]} y={y[i]}\nsim {sim_tr[i]:.2f}" for i in border],
                  title="Kept, just below the threshold (aliens here -> lower alien_sim_threshold)")
        plt.figure(figsize=(8, 3))
        plt.hist(sim_tr, 60, alpha=.6, density=True, label="train")
        plt.hist(sim_te, 60, alpha=.6, density=True, label="test")
        plt.axvline(thr, color="k", ls="--")
        plt.legend(); plt.title("max similarity to alien examples"); plt.show()
else:
    print("alien filter off: add example ids to CFG.alien_examples after looking at the gallery")

### 6.4 Label noise
A quick K-fold model gives out-of-fold probabilities. A training image is dropped when its given label
receives very low probability while another class is predicted with high confidence.

In [ ]:
stage(6, "cleaning: label noise")
noise_flag = np.zeros(len(X), dtype=bool)
if CFG.noise_filter:
    pool = np.where(keep)[0]
    oof_noise, _, _, _ = run_cv(CFG.noise_model, "noise", pool, CFG.noise_model["epochs"], "all",
                                predict_test=False)
    s, m = oof_score(oof_noise)
    print(f"\nnoise-model OOF weighted F1: {s:.4f}")
    idx = np.where(m)[0]
    p_given = oof_noise[idx, y[idx]]
    p_pred = oof_noise[idx].max(1)
    cand = idx[(p_given < CFG.noise_p_given_max) & (p_pred > CFG.noise_p_pred_min)]
    cand = cand[np.argsort(oof_noise[cand, y[cand]])][:int(CFG.noise_max_drop_frac * len(X))]
    noise_flag[cand] = True
    keep[cand] = False
    susp = pd.DataFrame({"id": train_df.id.values[idx], "label": y[idx], "pred": oof_noise[idx].argmax(1),
                         "p_given": p_given, "p_pred": p_pred}).sort_values("p_given")
    susp.to_csv(f"{OUT_DIR}/suspicious_labels.csv", index=False)
    report.update(noise_model_oof_wf1=float(s), label_noise_removed=int(len(cand)))
    sh = cand[:24]
    show_grid([X[i] for i in sh],
              [f"{train_df.id[i]}\ny={y[i]} pred={oof_noise[i].argmax()}" for i in sh],
              title="Removed as probable label noise")
    print(f"removed {len(cand)} probable label-noise images")

CLEAN_IDX = np.where(keep)[0]
train_df["kept"] = keep
train_df.to_csv(f"{OUT_DIR}/train_cleaning.csv", index=False)
report["n_train_clean"] = int(len(CLEAN_IDX))
print(f"\nclean training set: {len(CLEAN_IDX)} / {len(X)} images")
print(pd.Series(np.bincount(y[CLEAN_IDX], minlength=NUM_CLASSES), name="clean count").to_frame().T)

## 7. Stage A: K-fold training of every backbone on the clean set

In [ ]:
stage(7, "stage A: K-fold training of every backbone")
results = {}

def save_submission(probs, name):
    labels = probs.argmax(1).astype(int)
    if CFG.use_train_duplicates_for_test:
        labels = np.where(test_dup_label >= 0, test_dup_label, labels)
    sub = pd.DataFrame({"id": sub_df["id"], "label": labels})
    assert len(sub) == len(sub_df) and sub.label.between(0, NUM_CLASSES - 1).all()
    sub.to_csv(f"{OUT_DIR}/{name}", index=False)
    return sub

for mcfg in CFG.models:
    key = f"A_{short_name(mcfg['name'])}"
    if hours_left() < 0.5:
        print("time budget reached, skipping", key); continue
    oof, tp, n, hist = run_cv(mcfg, "A", CLEAN_IDX, mcfg["epochs"], CFG.folds_to_run)
    if n == 0:
        continue
    s, _ = oof_score(oof)
    results[key] = dict(oof=oof, test=tp, folds=n, cv=s)
    np.save(f"{OUT_DIR}/oof_{key}.npy", oof); np.save(f"{OUT_DIR}/test_{key}.npy", tp)
    print(f"\n>>> {key}: OOF weighted F1 {s:.4f} over {n} folds")

def ensemble(keys):
    with np.errstate(invalid="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        oof = np.nanmean(np.stack([results[k]["oof"] for k in keys]), axis=0)
    test = np.mean(np.stack([results[k]["test"] for k in keys]), axis=0)
    return oof, test

keys_a = [k for k in results if k.startswith("A_")]
oof_a, test_a = ensemble(keys_a)
cv_a, _ = oof_score(oof_a)
print(f"\nStage A ensemble ({', '.join(keys_a)}): OOF weighted F1 {cv_a:.4f}")
save_submission(test_a, "submission_stageA.csv")
np.save(f"{OUT_DIR}/test_probs_stageA.npy", test_a)
report["stageA"] = {k: round(float(results[k]["cv"]), 4) for k in keys_a} | {"ensemble": round(float(cv_a), 4)}

## 8. Stage B: pseudo-labelling
Confident test predictions of the stage A ensemble are added to training, so the models also learn the
appearance of the new observations. Validation folds stay purely on labelled training images.

In [ ]:
stage(8, "stage B: pseudo-labelling")
keys_b = []
if CFG.pseudo_label and hours_left() > 1.0:
    conf = test_a.max(1)
    pl = test_a.argmax(1)
    sel = np.where((conf >= CFG.pseudo_conf) & ~alien_test)[0]
    cap = int(CFG.pseudo_max_ratio * len(CLEAN_IDX))
    if len(sel) > cap:
        sel = sel[np.argsort(-conf[sel])][:cap]
    pseudo_items, pseudo_labels = make_items(1, sel), pl[sel]
    report["pseudo_labels"] = int(len(sel))
    print(f"pseudo-labelled test images: {len(sel)} / {len(XT)} (conf >= {CFG.pseudo_conf})")
    print(pd.Series(np.bincount(pseudo_labels, minlength=NUM_CLASSES), name="pseudo count").to_frame().T)

    for mcfg in CFG.models:
        key = f"B_{short_name(mcfg['name'])}"
        if hours_left() < 0.75:
            print("time budget reached, skipping", key); continue
        oof, tp, n, hist = run_cv(mcfg, "B", CLEAN_IDX, CFG.pseudo_epochs, CFG.folds_to_run,
                                  pseudo_items, pseudo_labels)
        if n == 0:
            continue
        s, _ = oof_score(oof)
        results[key] = dict(oof=oof, test=tp, folds=n, cv=s)
        np.save(f"{OUT_DIR}/oof_{key}.npy", oof); np.save(f"{OUT_DIR}/test_{key}.npy", tp)
        keys_b.append(key)
        print(f"\n>>> {key}: OOF weighted F1 {s:.4f} over {n} folds")
else:
    print("pseudo-labelling skipped")

## 9. Final ensemble, class-bias tuning and submission

In [ ]:
stage(9, "final ensemble and submission")
candidates = {"A": keys_a}
if keys_b:
    candidates["B"] = keys_b
    candidates["A+B"] = keys_a + keys_b
scores = {}
for name, keys in candidates.items():
    o, _ = ensemble(keys)
    scores[name] = oof_score(o, CLEAN_IDX)[0]
print("OOF weighted F1 by candidate:", {k: round(v, 4) for k, v in scores.items()})
best_name = max(scores, key=scores.get)
final_oof, final_test = ensemble(candidates[best_name])
print("selected:", best_name)

def tune_class_bias(probs, labels, grid=np.linspace(-0.6, 0.6, 13), rounds=2):
    logp = np.log(probs + 1e-9)
    bias = np.zeros(NUM_CLASSES)
    best = f1_score(labels, (logp + bias).argmax(1), average="weighted")
    for _ in range(rounds):
        for c in range(NUM_CLASSES):
            for g in grid:
                b = bias.copy(); b[c] = g
                s = f1_score(labels, (logp + b).argmax(1), average="weighted")
                if s > best + 1e-6:
                    best, bias = s, b
    return bias, best

_, m = oof_score(final_oof, CLEAN_IDX)
base_cv = weighted_f1(y[m], final_oof[m])
bias = np.zeros(NUM_CLASSES)
if CFG.optimize_class_bias and m.sum() > 200:
    # tuned on half of the OOF set and checked on the other half to avoid overfitting
    idx = np.where(m)[0]; rng = np.random.RandomState(CFG.seed); rng.shuffle(idx)
    h1, h2 = idx[: len(idx) // 2], idx[len(idx) // 2:]
    b1, _ = tune_class_bias(final_oof[h1], y[h1])
    gain = (f1_score(y[h2], (np.log(final_oof[h2] + 1e-9) + b1).argmax(1), average="weighted")
            - weighted_f1(y[h2], final_oof[h2]))
    if gain > 0.002:
        bias, _ = tune_class_bias(final_oof[idx], y[idx])
    print(f"class-bias held-out gain: {gain:+.4f} -> {'applied' if bias.any() else 'not applied'}")

final_test_adj = np.exp(np.log(final_test + 1e-9) + bias)
final_test_adj /= final_test_adj.sum(1, keepdims=True)
final_oof_adj = np.exp(np.log(final_oof[m] + 1e-9) + bias)
final_cv = weighted_f1(y[m], final_oof_adj)
print(f"\nFINAL local CV weighted F1: {final_cv:.4f} (before bias {base_cv:.4f})\n")
print(classification_report(y[m], final_oof_adj.argmax(1), labels=list(range(NUM_CLASSES)),
                            target_names=CLASS_NAMES, digits=3, zero_division=0))

with np.errstate(invalid="ignore"):
    cm = np.nan_to_num(confusion_matrix(y[m], final_oof_adj.argmax(1), labels=list(range(NUM_CLASSES)),
                                        normalize="true"))
plt.figure(figsize=(7, 6)); plt.imshow(cm, cmap="Blues"); plt.colorbar()
plt.xlabel("predicted"); plt.ylabel("true"); plt.title("Normalized confusion matrix (OOF, final ensemble)")
plt.xticks(range(NUM_CLASSES)); plt.yticks(range(NUM_CLASSES)); plt.show()

submission = save_submission(final_test_adj, "submission.csv")
log("submission.csv written")
np.save(f"{OUT_DIR}/test_probs_final.npy", final_test_adj)
report.update(final_selection=best_name, final_cv_wf1=round(float(final_cv), 4),
              class_bias=[round(float(b), 3) for b in bias],
              runtime_h=round((time.time() - RUN_START) / 3600, 2))
json.dump(report, open(f"{OUT_DIR}/run_report.json", "w"), indent=2)

print(json.dumps(report, indent=2))
print("\nPredicted class distribution on test:")
print(submission.label.value_counts(normalize=True).sort_index().round(3))
print(submission.head())